In [ ]:
# Cell 1: Install and imports
!pip install -q opencv-python-headless tqdm

from __future__ import annotations

import json
import math
import random
import re
import shutil
import warnings
from collections.abc import Callable
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm


In [ ]:
# Cell 2: Configuration
# Update these paths if your Kaggle dataset mounts use different folder names.
CONFIG = {
    "kvasir_root": "/kaggle/input/kvasir-seg/kvasir-seg",
    "clinicdb_root": "/kaggle/input/cvc-clinicdb",
    "colondb_root": "/kaggle/input/cvc-colondb/CVC-ColonDB",
    "isic_train_img": "/kaggle/input/isic-2018-task1/ISIC2018_Task1-2_Training_Input",
    "isic_train_mask": "/kaggle/input/isic-2018-task1/ISIC2018_Task1_Training_GroundTruth",
    "isic_test_img": "/kaggle/input/isic-2018-task1/ISIC2018_Task1-2_Test_Input",
    "isic_test_mask": "/kaggle/input/isic-2018-task1/ISIC2018_Task1_Test_GroundTruth",
    "output_root": "/kaggle/working/WBSNet_Dataset",
    "image_size": 352,
    "seed": 42,
    "kvasir_train": 880,
    "kvasir_val": 120,
    "clinicdb_train": 550,
    "clinicdb_val": 62,
    "isic_val_frac": 0.10,
}

BOUNDARY_DILATED = True
EXPECTED_COUNTS = {
    ("kvasir", "train"): 880,
    ("kvasir", "val"): 120,
    ("cvc_clinicdb", "train"): 550,
    ("cvc_clinicdb", "val"): 62,
    ("cvc_colondb", "test"): 380,
    ("isic2018", "train"): 2334,
    ("isic2018", "val"): 260,
    ("isic2018", "test"): 1000,
}

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
DEFAULT_KAGGLE_OUTPUT_ROOT = Path("/kaggle/working/WBSNet_Dataset")
REQUESTED_OUTPUT_ROOT = Path(CONFIG["output_root"]).expanduser()

if REQUESTED_OUTPUT_ROOT == KAGGLE_INPUT_ROOT or KAGGLE_INPUT_ROOT in REQUESTED_OUTPUT_ROOT.parents:
    warnings.warn(
        "CONFIG['output_root'] points inside Kaggle's read-only /kaggle/input mount. "
        f"Switching output_root to {DEFAULT_KAGGLE_OUTPUT_ROOT}."
    )
    OUTPUT_ROOT = DEFAULT_KAGGLE_OUTPUT_ROOT
else:
    OUTPUT_ROOT = REQUESTED_OUTPUT_ROOT

OUTPUT_ROOT = OUTPUT_ROOT.resolve()
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

# Remove old outputs on rerun so counts never include stale files.
try:
    if OUTPUT_ROOT.exists():
        shutil.rmtree(OUTPUT_ROOT)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
except OSError as exc:
    raise OSError(
        f"Unable to prepare output_root={OUTPUT_ROOT}. On Kaggle, this path must live under /kaggle/working."
    ) from exc

print(f"Output root: {OUTPUT_ROOT}")


In [ ]:
# Cell 3: Utility functions
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
MASK_EXTENSIONS = IMAGE_EXTENSIONS
SPLIT_ORDER = {"train": 0, "val": 1, "test": 2}
RUN_SUMMARY: dict[str, dict[str, dict[str, object]]] = {}


def natural_sort_key(value: str) -> list[object]:
    return [int(token) if token.isdigit() else token.lower() for token in re.split(r"(\\d+)", value)]


def resolve_candidate_dir(root: str | Path, candidates: list[str]) -> Path:
    root = Path(root)
    for candidate in candidates:
        path = root / candidate
        if path.exists() and path.is_dir():
            return path
    raise FileNotFoundError(f"Could not find any of {candidates} under {root}")


def get_sorted_files(directory: str | Path, suffixes: set[str]) -> list[Path]:
    directory = Path(directory)
    return sorted(
        [path for path in directory.iterdir() if path.is_file() and path.suffix.lower() in suffixes],
        key=lambda path: natural_sort_key(path.name),
    )


def build_file_map(
    directory: str | Path,
    suffixes: set[str],
    key_fn: Callable[[Path], str] | None = None,
) -> dict[str, Path]:
    files: dict[str, Path] = {}
    for path in get_sorted_files(directory, suffixes):
        key = key_fn(path) if key_fn else path.stem
        if key in files:
            warnings.warn(f"Duplicate key '{key}' found in {directory}; keeping the last file.")
        files[key] = path
    return files


def build_pairs(
    image_dir: str | Path,
    mask_dir: str | Path,
    image_key_fn: Callable[[Path], str] | None = None,
    mask_key_fn: Callable[[Path], str] | None = None,
) -> list[dict[str, object]]:
    image_map = build_file_map(image_dir, IMAGE_EXTENSIONS, image_key_fn)
    mask_map = build_file_map(mask_dir, MASK_EXTENSIONS, mask_key_fn)

    missing_masks = sorted(set(image_map) - set(mask_map), key=natural_sort_key)
    missing_images = sorted(set(mask_map) - set(image_map), key=natural_sort_key)
    if missing_masks:
        warnings.warn(
            f"{Path(image_dir).name}: {len(missing_masks)} images do not have masks and will be skipped."
        )
    if missing_images:
        warnings.warn(
            f"{Path(mask_dir).name}: {len(missing_images)} masks do not have images and will be skipped."
        )

    shared_keys = sorted(set(image_map) & set(mask_map), key=natural_sort_key)
    return [
        {"sample_id": key, "image_path": image_map[key], "mask_path": mask_map[key]}
        for key in shared_keys
    ]


def take_fixed_split(
    records: list[dict[str, object]],
    train_count: int,
    val_count: int,
    dataset_name: str,
) -> tuple[list[dict[str, object]], list[dict[str, object]]]:
    ordered = sorted(records, key=lambda record: natural_sort_key(str(record["sample_id"])))
    expected_total = train_count + val_count
    if len(ordered) != expected_total:
        warnings.warn(f"{dataset_name}: expected {expected_total} paired samples, found {len(ordered)}.")
    train_records = ordered[: min(train_count, len(ordered))]
    val_records = ordered[len(train_records) : len(train_records) + val_count]
    return train_records, val_records


def take_seeded_val_split(
    records: list[dict[str, object]],
    val_fraction: float,
    seed: int,
) -> tuple[list[dict[str, object]], list[dict[str, object]]]:
    ordered = sorted(records, key=lambda record: natural_sort_key(str(record["sample_id"])))
    if not ordered:
        return [], []
    shuffled = ordered.copy()
    random.Random(seed).shuffle(shuffled)
    val_count = max(1, math.ceil(len(shuffled) * val_fraction))
    train_records = shuffled[:-val_count]
    val_records = shuffled[-val_count:]
    return train_records, val_records


def make_split_dirs(dataset_name: str, split_name: str) -> dict[str, Path]:
    split_root = OUTPUT_ROOT / dataset_name / split_name
    split_dirs = {
        "images": split_root / "images",
        "masks": split_root / "masks",
        "boundaries": split_root / "boundaries",
    }
    for path in split_dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return split_dirs


def read_rgb_image(path: str | Path) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f"Unable to read image: {path}")
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def read_mask(path: str | Path) -> np.ndarray:
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f"Unable to read mask: {path}")
    return mask


def resize_image(image: np.ndarray, image_size: int) -> np.ndarray:
    return cv2.resize(image, (image_size, image_size), interpolation=cv2.INTER_LINEAR)


def resize_mask(mask: np.ndarray, image_size: int) -> np.ndarray:
    return cv2.resize(mask, (image_size, image_size), interpolation=cv2.INTER_NEAREST)


def binarize_mask(mask: np.ndarray) -> np.ndarray:
    return np.where(mask > 127, 255, 0).astype(np.uint8)


def generate_boundary_gt(mask: np.ndarray, dilate: bool = True) -> np.ndarray:
    mask_float = (mask / 255.0).astype(np.float32)
    sobel_x = cv2.Sobel(mask_float, cv2.CV_32F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(mask_float, cv2.CV_32F, 0, 1, ksize=3)
    edge = np.sqrt(np.square(sobel_x) + np.square(sobel_y))
    edge = (edge > 0).astype(np.uint8) * 255
    if dilate:
        kernel = np.ones((3, 3), np.uint8)
        edge = cv2.dilate(edge, kernel, iterations=1)
    return edge


def save_image(image_rgb: np.ndarray, save_path: str | Path) -> None:
    cv2.imwrite(str(save_path), cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))


def save_mask(mask: np.ndarray, save_path: str | Path) -> None:
    cv2.imwrite(str(save_path), mask)


def save_boundary(boundary: np.ndarray, save_path: str | Path) -> None:
    cv2.imwrite(str(save_path), boundary)


def count_pngs(directory: str | Path) -> int:
    return len(list(Path(directory).glob("*.png")))


def register_summary(dataset_name: str, split_name: str, summary: dict[str, object]) -> None:
    RUN_SUMMARY.setdefault(dataset_name, {})[split_name] = summary


def process_records(records: list[dict[str, object]], dataset_name: str, split_name: str) -> dict[str, object]:
    split_dirs = make_split_dirs(dataset_name, split_name)
    channel_sum = np.zeros(3, dtype=np.float64)
    channel_sq_sum = np.zeros(3, dtype=np.float64)
    pixel_count = 0
    skipped = 0

    for record in tqdm(records, desc=f"{dataset_name}/{split_name}", unit="sample"):
        sample_id = str(record["sample_id"])
        image_path = Path(record["image_path"])
        mask_path = Path(record["mask_path"])

        try:
            image = resize_image(read_rgb_image(image_path), CONFIG["image_size"])
            mask = binarize_mask(resize_mask(read_mask(mask_path), CONFIG["image_size"]))
        except FileNotFoundError as exc:
            warnings.warn(f"{sample_id}: {exc}. Skipping sample.")
            skipped += 1
            continue

        boundary = generate_boundary_gt(mask, dilate=BOUNDARY_DILATED)
        output_name = f"{sample_id}.png"

        save_image(image, split_dirs["images"] / output_name)
        save_mask(mask, split_dirs["masks"] / output_name)
        save_boundary(boundary, split_dirs["boundaries"] / output_name)

        image_float = image.astype(np.float64) / 255.0
        flat = image_float.reshape(-1, 3)
        channel_sum += flat.sum(axis=0)
        channel_sq_sum += np.square(flat).sum(axis=0)
        pixel_count += flat.shape[0]

    n_images = count_pngs(split_dirs["images"])
    n_masks = count_pngs(split_dirs["masks"])
    n_boundaries = count_pngs(split_dirs["boundaries"])
    mean = channel_sum / pixel_count if pixel_count else np.zeros(3, dtype=np.float64)
    variance = channel_sq_sum / pixel_count - np.square(mean) if pixel_count else np.zeros(3, dtype=np.float64)
    variance = np.maximum(variance, 0.0)
    summary = {
        "n_images": int(n_images),
        "n_masks": int(n_masks),
        "n_boundaries": int(n_boundaries),
        "image_size": [CONFIG["image_size"], CONFIG["image_size"]],
        "boundary_dilated": bool(BOUNDARY_DILATED),
        "image_mean_rgb": [round(float(value), 6) for value in mean],
        "image_std_rgb": [round(float(value), 6) for value in np.sqrt(variance)],
        "skipped_samples": int(skipped),
    }
    register_summary(dataset_name, split_name, summary)
    print(f"Completed {dataset_name}/{split_name}: {n_images} samples saved, {skipped} skipped.")
    return summary


def collect_dataset_info() -> dict[str, object]:
    dataset_info: dict[str, object] = {
        "seed": CONFIG["seed"],
        "image_size": [CONFIG["image_size"], CONFIG["image_size"]],
        "boundary_dilated": BOUNDARY_DILATED,
        "datasets": {},
    }
    for dataset_name in sorted(RUN_SUMMARY):
        dataset_info["datasets"][dataset_name] = {}
        for split_name in sorted(RUN_SUMMARY[dataset_name], key=lambda split: SPLIT_ORDER.get(split, 99)):
            split_root = OUTPUT_ROOT / dataset_name / split_name
            counts = {
                "n_images": count_pngs(split_root / "images"),
                "n_masks": count_pngs(split_root / "masks"),
                "n_boundaries": count_pngs(split_root / "boundaries"),
            }
            dataset_info["datasets"][dataset_name][split_name] = {
                **counts,
                "image_size": RUN_SUMMARY[dataset_name][split_name]["image_size"],
                "boundary_dilated": RUN_SUMMARY[dataset_name][split_name]["boundary_dilated"],
                "image_mean_rgb": RUN_SUMMARY[dataset_name][split_name]["image_mean_rgb"],
                "image_std_rgb": RUN_SUMMARY[dataset_name][split_name]["image_std_rgb"],
                "skipped_samples": RUN_SUMMARY[dataset_name][split_name]["skipped_samples"],
            }
    return dataset_info


def verify_split(dataset_name: str, split_name: str, rng: random.Random) -> None:
    split_root = OUTPUT_ROOT / dataset_name / split_name
    image_paths = get_sorted_files(split_root / "images", {".png"})
    mask_paths = get_sorted_files(split_root / "masks", {".png"})
    boundary_paths = get_sorted_files(split_root / "boundaries", {".png"})

    print(f"[{dataset_name}/{split_name}] images={len(image_paths)}, masks={len(mask_paths)}, boundaries={len(boundary_paths)}")
    assert len(image_paths) == len(mask_paths) == len(boundary_paths), f"Count mismatch for {dataset_name}/{split_name}"

    expected = EXPECTED_COUNTS.get((dataset_name, split_name))
    if expected is not None:
        print(f"Expected image count: {expected}")

    if not image_paths:
        return

    sample_path = rng.choice(image_paths)
    sample_id = sample_path.stem
    image = read_rgb_image(sample_path)
    mask = read_mask(split_root / "masks" / f"{sample_id}.png")
    boundary = read_mask(split_root / "boundaries" / f"{sample_id}.png")

    assert set(np.unique(mask)).issubset({0, 255}), f"Mask has non-binary values for {sample_id}"
    assert set(np.unique(boundary)).issubset({0, 255}), f"Boundary has non-binary values for {sample_id}"
    print(f"Mask min/max: {int(mask.min())}/{int(mask.max())}")
    print(f"Boundary min/max: {int(boundary.min())}/{int(boundary.max())}")

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(image)
    axes[0].set_title(f"{dataset_name}/{split_name}\\n{sample_id}")
    axes[1].imshow(mask, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title("Mask")
    axes[2].imshow(boundary, cmap="gray", vmin=0, vmax=255)
    axes[2].set_title("Boundary")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
# Cell 4: Process Kvasir-SEG
kvasir_root = Path(CONFIG["kvasir_root"])
kvasir_image_dir = resolve_candidate_dir(kvasir_root, ["images"])
kvasir_mask_dir = resolve_candidate_dir(kvasir_root, ["masks"])

kvasir_records = build_pairs(kvasir_image_dir, kvasir_mask_dir)
print(f"Discovered {len(kvasir_records)} Kvasir pairs.")

kvasir_train_records, kvasir_val_records = take_fixed_split(
    kvasir_records,
    train_count=CONFIG["kvasir_train"],
    val_count=CONFIG["kvasir_val"],
    dataset_name="kvasir",
)

process_records(kvasir_train_records, "kvasir", "train")
process_records(kvasir_val_records, "kvasir", "val")


In [ ]:
# Cell 5: Process CVC-ClinicDB
clinic_root = Path(CONFIG["clinicdb_root"])
clinic_image_dir = resolve_candidate_dir(clinic_root, ["Original", "PNG/Original"])
clinic_mask_dir = resolve_candidate_dir(
    clinic_root,
    ["Ground Truth", "GroundTruth", "Ground_Truth", "PNG/Ground Truth", "PNG/GroundTruth", "PNG/Ground_Truth"],
)
print(f"Using ClinicDB image dir: {clinic_image_dir}")
print(f"Using ClinicDB mask dir:  {clinic_mask_dir}")

clinic_records = build_pairs(clinic_image_dir, clinic_mask_dir)
print(f"Discovered {len(clinic_records)} ClinicDB pairs.")

clinic_train_records, clinic_val_records = take_fixed_split(
    clinic_records,
    train_count=CONFIG["clinicdb_train"],
    val_count=CONFIG["clinicdb_val"],
    dataset_name="cvc_clinicdb",
)

process_records(clinic_train_records, "cvc_clinicdb", "train")
process_records(clinic_val_records, "cvc_clinicdb", "val")


In [ ]:
# Cell 6: Process CVC-ColonDB
colondb_root = Path(CONFIG["colondb_root"])
colondb_image_dir = resolve_candidate_dir(colondb_root, ["images"])
colondb_mask_dir = resolve_candidate_dir(colondb_root, ["masks"])

colondb_records = build_pairs(colondb_image_dir, colondb_mask_dir)
print(f"Discovered {len(colondb_records)} ColonDB pairs.")

process_records(colondb_records, "cvc_colondb", "test")


In [ ]:
# Cell 7: Process ISIC 2018 Task 1
isic_mask_key = lambda path: path.stem.replace("_segmentation", "")

isic_train_records = build_pairs(
    CONFIG["isic_train_img"],
    CONFIG["isic_train_mask"],
    mask_key_fn=isic_mask_key,
)
print(f"Discovered {len(isic_train_records)} ISIC train pairs.")

isic_train_split, isic_val_split = take_seeded_val_split(
    isic_train_records,
    val_fraction=CONFIG["isic_val_frac"],
    seed=CONFIG["seed"],
)
print(f"ISIC train split: {len(isic_train_split)}")
print(f"ISIC val split:   {len(isic_val_split)}")

process_records(isic_train_split, "isic2018", "train")
process_records(isic_val_split, "isic2018", "val")

isic_test_records = build_pairs(
    CONFIG["isic_test_img"],
    CONFIG["isic_test_mask"],
    mask_key_fn=isic_mask_key,
)
print(f"Discovered {len(isic_test_records)} ISIC test pairs.")

process_records(isic_test_records, "isic2018", "test")


In [ ]:
# Cell 8: Generate dataset_info.json
dataset_info = collect_dataset_info()
dataset_info_path = OUTPUT_ROOT / "dataset_info.json"
dataset_info_path.write_text(json.dumps(dataset_info, indent=2), encoding="utf-8")

print(f"Saved dataset metadata to {dataset_info_path}")
dataset_info


In [ ]:
# Cell 9: Verification
verification_rng = random.Random(CONFIG["seed"])

for dataset_name in ["kvasir", "cvc_clinicdb", "cvc_colondb", "isic2018"]:
    if dataset_name not in RUN_SUMMARY:
        continue
    for split_name in sorted(RUN_SUMMARY[dataset_name], key=lambda split: SPLIT_ORDER.get(split, 99)):
        verify_split(dataset_name, split_name, verification_rng)


In [ ]:
# Cell 10: Zip and save
archive_path = shutil.make_archive(
    str(OUTPUT_ROOT),
    "zip",
    str(OUTPUT_ROOT.parent),
    OUTPUT_ROOT.name,
)
print(f"Done! Download {archive_path} from Kaggle output.")
